# Trabalho Prático — Otimização Paralela de GEMM

**Disciplina:** Arquitetura de Computadores  
**Tema:** Processamento paralelo em CPU e GPU  
**Problema:** Multiplicação de matrizes densas — GEMM

Neste trabalho, você irá partir de uma implementação ingênua de multiplicação de matrizes e evoluir para versões otimizadas usando:

1. melhoria de localidade de memória;
2. blocagem/tiling;
3. vetorização SIMD;
4. paralelismo com OpenMP;
5. aceleração em GPU com CUDA;
6. análise completa de desempenho.

A operação calculada é:

$C = A \times B$


Para matrizes quadradas $N \times N$, o número aproximado de operações de ponto flutuante é:

$2N^3$

Logo, o desempenho em GFLOP/s é:

$
GFLOP/s = \frac{2N^3}{tempo \times 10^9}
$

## Entregáveis

Ao final, cada grupo deve entregar:

1. Este notebook preenchido com os resultados em PDF como anexo junto ao relatório.
2. Relatório em PDF contendo:
   - Metodologia experimental;
   - Tabelas de tempo;
   - Gráficos de desempenho;
   - Cálculo de GFLOP/s, speedup e eficiência;
   - Discussão dos resultados;
   - Conclusão crítica.

## Regras

- Não usar BLAS, cuBLAS, Eigen ou bibliotecas equivalentes na parte principal.
- Cada experimento deve ser executado pelo menos 5 vezes.
- Use a média dos tempos.
- Valide todas as versões comparando com a versão sequencial de referência.
- Bibliotecas prontas podem ser usadas apenas como comparação bônus.
- Use o colab para execução em GPU, mas para os testes CPU tente usar sua máquina devido o colab não oferecer mais de 2 nucleos de CPU.

# Parte 0 — Configuração do ambiente

Execute as células abaixo para verificar a CPU, a GPU e os compiladores disponíveis no Colab.

> Para usar CUDA, ative GPU em:  
> **Ambiente de execução → Alterar tipo de ambiente de execução → GPU**.

In [ ]:
!lscpu | head -n 30

In [ ]:
!nvidia-smi || true

In [ ]:
!gcc --version | head -n 1
!nvcc --version | tail -n 4 || true

# Parte 1 — Código C base

A próxima célula cria um programa em C contendo várias versões de GEMM em CPU:

- `naive`: implementação ingênua;
- `transposed`: usa a matriz B transposta;
- `blocked`: usa blocagem/tiling;
- `openmp`: usa OpenMP;
- `blocked_openmp`: combina blocagem com OpenMP.

O programa também mede tempo, calcula GFLOP/s e valida o resultado.

Nesta etapa você deve implementar apenas as funções marcadas com **TODO**.
Caso necessário crie funções auxiliares, porém não mude as assinaturas das funções.

In [ ]:
%%writefile gemm_cpu.c

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <time.h>
#include <omp.h>

#ifndef ALIGNMENT
#define ALIGNMENT 64
#endif

static double now_seconds() {
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return ts.tv_sec + ts.tv_nsec * 1e-9;
}

static float rand_float() {
    return (float)rand() / (float)RAND_MAX;
}

static void fill_matrix(int N, float *M) {
    for (int i = 0; i < N*N; i++) {
        M[i] = rand_float();
    }
}

static void zero_matrix(int N, float *M) {
    memset(M, 0, sizeof(float) * N * N);
}

static double max_abs_error(int N, const float *A, const float *B) {
    double err = 0.0;
    for (int i = 0; i < N*N; i++) {
        double e = fabs((double)A[i] - (double)B[i]);
        if (e > err) err = e;
    }
    return err;
}

void gemm_naive(int N, const float *A, const float *B, float *C) {
    // Versao ingenua: ordem de lacos i-j-k.
    // O laco interno percorre B por colunas (B[k*N + j]), com passo N,
    // o que causa baixa localidade de cache e pessimo desempenho.
    for (int i = 0; i < N; i++) {
        for (int j = 0; j < N; j++) {
            float sum = 0.0f;
            for (int k = 0; k < N; k++) {
                sum += A[i*N + k] * B[k*N + j];
            }
            C[i*N + j] = sum;
        }
    }
}

void transpose(int N, const float *B, float *BT) {
    for (int i = 0; i < N; i++) {
        for (int j = 0; j < N; j++) {
            BT[j*N + i] = B[i*N + j];
        }
    }
}

void gemm_transposed(int N, const float *A, const float *BT, float *C) {
    // Usa B transposta (BT). Agora o laco interno percorre A e BT
    // sequencialmente na memoria (A[i*N + k] e BT[j*N + k]), melhorando
    // a localidade espacial e permitindo vetorizacao do produto-soma.
    for (int i = 0; i < N; i++) {
        for (int j = 0; j < N; j++) {
            float sum = 0.0f;
            for (int k = 0; k < N; k++) {
                sum += A[i*N + k] * BT[j*N + k];
            }
            C[i*N + j] = sum;
        }
    }
}

void gemm_blocked(int N, int BS, const float *A, const float *B, float *C) {
    // Blocagem/tiling: processa submatrizes BS x BS para reutilizar dados
    // que cabem no cache. Usa a ordem i-k-j no nucleo, de modo que o laco
    // interno (j) percorre B e C sequencialmente e pode ser vetorizado.
    // Assume C previamente zerada (feito em main via zero_matrix).
    for (int ii = 0; ii < N; ii += BS) {
        int i_max = (ii + BS < N) ? ii + BS : N;
        for (int kk = 0; kk < N; kk += BS) {
            int k_max = (kk + BS < N) ? kk + BS : N;
            for (int jj = 0; jj < N; jj += BS) {
                int j_max = (jj + BS < N) ? jj + BS : N;
                for (int i = ii; i < i_max; i++) {
                    for (int k = kk; k < k_max; k++) {
                        float a = A[i*N + k];
                        for (int j = jj; j < j_max; j++) {
                            C[i*N + j] += a * B[k*N + j];
                        }
                    }
                }
            }
        }
    }
}

void gemm_openmp(int N, const float *A, const float *B, float *C) {
    // Paralelizacao com OpenMP. Cada thread processa um conjunto de linhas i
    // de C (linhas disjuntas => sem condicao de corrida na escrita).
    // Ordem i-k-j para boa localidade e vetorizacao do laco interno.
    // Assume C previamente zerada (feito em main via zero_matrix).
    #pragma omp parallel for schedule(static)
    for (int i = 0; i < N; i++) {
        for (int k = 0; k < N; k++) {
            float a = A[i*N + k];
            for (int j = 0; j < N; j++) {
                C[i*N + j] += a * B[k*N + j];
            }
        }
    }
}

void gemm_blocked_openmp(int N, int BS, const float *A, const float *B, float *C) {
    // Combina blocagem com OpenMP. A paralelizacao ocorre sobre os blocos de
    // linhas (ii); como cada bloco ii escreve em linhas distintas de C, nao ha
    // condicao de corrida. Internamente reaproveita o tiling para o cache.
    // Assume C previamente zerada (feito em main via zero_matrix).
    #pragma omp parallel for schedule(static)
    for (int ii = 0; ii < N; ii += BS) {
        int i_max = (ii + BS < N) ? ii + BS : N;
        for (int kk = 0; kk < N; kk += BS) {
            int k_max = (kk + BS < N) ? kk + BS : N;
            for (int jj = 0; jj < N; jj += BS) {
                int j_max = (jj + BS < N) ? jj + BS : N;
                for (int i = ii; i < i_max; i++) {
                    for (int k = kk; k < k_max; k++) {
                        float a = A[i*N + k];
                        for (int j = jj; j < j_max; j++) {
                            C[i*N + j] += a * B[k*N + j];
                        }
                    }
                }
            }
        }
    }
}

static void usage(const char *prog) {
    printf("Usage: %s <version> <N> <repeats> [BS]\n", prog);
    printf("Versions: naive, transposed, blocked, openmp, blocked_openmp\n");
}

int main(int argc, char **argv) {
    if (argc < 4) {
        usage(argv[0]);
        return 1;
    }

    const char *version = argv[1];
    int N = atoi(argv[2]);
    int repeats = atoi(argv[3]);
    int BS = (argc >= 5) ? atoi(argv[4]) : 32;

    srand(0);

    float *A, *B, *BT, *C, *Cref;
    posix_memalign((void**)&A, ALIGNMENT, sizeof(float)*N*N);
    posix_memalign((void**)&B, ALIGNMENT, sizeof(float)*N*N);
    posix_memalign((void**)&BT, ALIGNMENT, sizeof(float)*N*N);
    posix_memalign((void**)&C, ALIGNMENT, sizeof(float)*N*N);
    posix_memalign((void**)&Cref, ALIGNMENT, sizeof(float)*N*N);

    fill_matrix(N, A);
    fill_matrix(N, B);
    zero_matrix(N, C);
    zero_matrix(N, Cref);

    // Referência para validação.
    // Para N grande, isto pode demorar. Ainda assim, é importante para corretude.
    gemm_naive(N, A, B, Cref);

    if (strcmp(version, "transposed") == 0) {
        transpose(N, B, BT);
    }

    double best = 1e30;
    double total = 0.0;

    for (int r = 0; r < repeats; r++) {
        zero_matrix(N, C);
        double t0 = now_seconds();

        if (strcmp(version, "naive") == 0) {
            gemm_naive(N, A, B, C);
        } else if (strcmp(version, "transposed") == 0) {
            gemm_transposed(N, A, BT, C);
        } else if (strcmp(version, "blocked") == 0) {
            gemm_blocked(N, BS, A, B, C);
        } else if (strcmp(version, "openmp") == 0) {
            gemm_openmp(N, A, B, C);
        } else if (strcmp(version, "blocked_openmp") == 0) {
            gemm_blocked_openmp(N, BS, A, B, C);
        } else {
            usage(argv[0]);
            return 1;
        }

        double t1 = now_seconds();
        double elapsed = t1 - t0;
        if (elapsed < best) best = elapsed;
        total += elapsed;
    }

    double avg = total / repeats;
    double gflops = (2.0 * N * N * N) / (avg * 1e9);
    double err = max_abs_error(N, C, Cref);

    printf("version,N,BS,threads,repeats,avg_time_s,best_time_s,GFLOPS,max_abs_error\n");
    printf("%s,%d,%d,%d,%d,%.6f,%.6f,%.6f,%.8f\n",
           version, N, BS, omp_get_max_threads(), repeats, avg, best, gflops, err);

    free(A); free(B); free(BT); free(C); free(Cref);
    return 0;
}

# Parte 2 — Compilação da versão CPU

A flag `-O3` ativa otimizações agressivas.  
A flag `-march=native` permite usar extensões específicas da CPU disponível.  
A flag `-fopenmp` habilita OpenMP.

A flag `-fopt-info-vec-optimized` pede ao compilador informações sobre vetorização automática.

In [ ]:
!gcc gemm_cpu.c -O3 -march=native -fopenmp -fopt-info-vec-optimized -o gemm_cpu 2> vec_report.txt
!cat vec_report.txt | head -n 40

# Parte 3 — Execução automatizada dos experimentos CPU

A próxima célula executa vários experimentos e salva os resultados em `cpu_results.csv`.

Ajuste os valores de `N`, `BS` e `threads` caso necessário.

In [ ]:
import subprocess
import pandas as pd
from io import StringIO

experiments = []
Ns = [128, 256, 512]
repeats = 5

for N in Ns:
    experiments.append(("naive", N, None, None))
    experiments.append(("transposed", N, None, None))
    for BS in [16, 32, 64]:
        experiments.append(("blocked", N, BS, None))
    for th in [1, 2, 4, 8]:
        experiments.append(("openmp", N, None, th))
        experiments.append(("blocked_openmp", N, 32, th))

rows = []
for version, N, BS, th in experiments:
    cmd = ["./gemm_cpu", version, str(N), str(repeats)]
    if BS is not None:
        cmd.append(str(BS))
    env = None
    if th is not None:
        env = {"OMP_NUM_THREADS": str(th)}
    result = subprocess.run(cmd, capture_output=True, text=True, env=env)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        continue
    lines = result.stdout.strip().splitlines()
    if len(lines) >= 2:
        df = pd.read_csv(StringIO("\n".join(lines[-2:])))
        rows.append(df)

cpu_results = pd.concat(rows, ignore_index=True)
cpu_results.to_csv("cpu_results.csv", index=False)
cpu_results

# Parte 4 — Gráficos CPU

Use os gráficos abaixo como ponto de partida. Você pode criar gráficos adicionais no relatório.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

cpu_results = pd.read_csv("cpu_results.csv")
cpu_results

In [ ]:
for N in sorted(cpu_results["N"].unique()):
    df = cpu_results[cpu_results["N"] == N].copy()
    labels = df["version"] + "_T" + df["threads"].astype(str) + "_BS" + df["BS"].astype(str)
    plt.figure(figsize=(12, 5))
    plt.bar(labels, df["avg_time_s"])
    plt.xticks(rotation=90)
    plt.ylabel("Tempo médio (s)")
    plt.title(f"Tempo médio por versão — N={N}")
    plt.tight_layout()
    plt.show()

In [ ]:
for N in sorted(cpu_results["N"].unique()):
    df = cpu_results[cpu_results["N"] == N].copy()
    labels = df["version"] + "_T" + df["threads"].astype(str) + "_BS" + df["BS"].astype(str)
    plt.figure(figsize=(12, 5))
    plt.bar(labels, df["GFLOPS"])
    plt.xticks(rotation=90)
    plt.ylabel("GFLOP/s")
    plt.title(f"Desempenho por versão — N={N}")
    plt.tight_layout()
    plt.show()

# Parte 5 — Código CUDA

A próxima célula deve-se implementar o programa em CUDA com duas versões:

- `cuda_naive`: uma thread calcula um elemento de C;
- `cuda_tiled`: usa memória compartilhada para reduzir acessos à memória global.

O programa mede separadamente:

1. tempo de cópia host → device;
2. tempo do kernel;
3. tempo de cópia device → host;
4. tempo total.

In [ ]:
%%writefile gemm_cuda.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define TILE 16

static float rand_float() {
    return (float)rand() / (float)RAND_MAX;
}

static void fill_matrix(int N, float *M) {
    for (int i = 0; i < N*N; i++) M[i] = rand_float();
}

static void gemm_cpu_ref(int N, const float *A, const float *B, float *C) {
    for (int i = 0; i < N; i++) {
        for (int j = 0; j < N; j++) {
            float sum = 0.0f;
            for (int k = 0; k < N; k++) {
                sum += A[i*N + k] * B[k*N + j];
            }
            C[i*N + j] = sum;
        }
    }
}

static double max_abs_error(int N, const float *A, const float *B) {
    double err = 0.0;
    for (int i = 0; i < N*N; i++) {
        double e = fabs((double)A[i] - (double)B[i]);
        if (e > err) err = e;
    }
    return err;
}

__global__ void gemm_cuda_naive(int N, const float *A, const float *B, float *C) {
    // Cada thread calcula um unico elemento C[row][col].
    // Mapeia (row, col) a partir dos indices de bloco e thread.
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < N && col < N) {
        float sum = 0.0f;
        for (int k = 0; k < N; k++) {
            sum += A[row*N + k] * B[k*N + col];
        }
        C[row*N + col] = sum;
    }
}

__global__ void gemm_cuda_tiled(int N, const float *A, const float *B, float *C) {
    // Versao com memoria compartilhada (tiling). Cada bloco carrega tiles
    // TILE x TILE de A e B para a memoria compartilhada, reduzindo os acessos
    // a memoria global e reutilizando os dados entre as threads do bloco.
    __shared__ float As[TILE][TILE];
    __shared__ float Bs[TILE][TILE];

    int tx = threadIdx.x;
    int ty = threadIdx.y;
    int row = blockIdx.y * TILE + ty;
    int col = blockIdx.x * TILE + tx;

    float sum = 0.0f;
    int numTiles = (N + TILE - 1) / TILE;

    for (int t = 0; t < numTiles; t++) {
        int aCol = t * TILE + tx;   // coluna de A para este tile
        int bRow = t * TILE + ty;   // linha de B para este tile

        // Carrega os tiles na memoria compartilhada (com guarda de borda).
        As[ty][tx] = (row < N && aCol < N) ? A[row*N + aCol] : 0.0f;
        Bs[ty][tx] = (bRow < N && col < N) ? B[bRow*N + col] : 0.0f;

        __syncthreads();

        // Produto parcial usando os dados ja em memoria compartilhada.
        for (int k = 0; k < TILE; k++) {
            sum += As[ty][k] * Bs[k][tx];
        }

        __syncthreads();
    }

    if (row < N && col < N) {
        C[row*N + col] = sum;
    }
}

static void check_cuda(cudaError_t err, const char *msg) {
    if (err != cudaSuccess) {
        fprintf(stderr, "CUDA error at %s: %s\n", msg, cudaGetErrorString(err));
        exit(1);
    }
}

int main(int argc, char **argv) {
    if (argc < 4) {
        printf("Usage: %s <naive|tiled> <N> <repeats>\n", argv[0]);
        return 1;
    }

    const char *version = argv[1];
    int N = atoi(argv[2]);
    int repeats = atoi(argv[3]);
    size_t bytes = sizeof(float) * N * N;

    srand(0);
    float *h_A = (float*)malloc(bytes);
    float *h_B = (float*)malloc(bytes);
    float *h_C = (float*)malloc(bytes);
    float *h_ref = (float*)malloc(bytes);

    fill_matrix(N, h_A);
    fill_matrix(N, h_B);

    gemm_cpu_ref(N, h_A, h_B, h_ref);

    float *d_A, *d_B, *d_C;
    check_cuda(cudaMalloc(&d_A, bytes), "cudaMalloc A");
    check_cuda(cudaMalloc(&d_B, bytes), "cudaMalloc B");
    check_cuda(cudaMalloc(&d_C, bytes), "cudaMalloc C");

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    float h2d_ms_total = 0.0f;
    float kernel_ms_total = 0.0f;
    float d2h_ms_total = 0.0f;

    dim3 block(TILE, TILE);
    dim3 grid((N + TILE - 1) / TILE, (N + TILE - 1) / TILE);

    for (int r = 0; r < repeats; r++) {
        float ms;

        cudaEventRecord(start);
        check_cuda(cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice), "copy A H2D");
        check_cuda(cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice), "copy B H2D");
        cudaEventRecord(stop);
        cudaEventSynchronize(stop);
        cudaEventElapsedTime(&ms, start, stop);
        h2d_ms_total += ms;

        cudaEventRecord(start);
        if (version[0] == 'n') {
            gemm_cuda_naive<<<grid, block>>>(N, d_A, d_B, d_C);
        } else {
            gemm_cuda_tiled<<<grid, block>>>(N, d_A, d_B, d_C);
        }
        check_cuda(cudaGetLastError(), "kernel launch");
        cudaEventRecord(stop);
        cudaEventSynchronize(stop);
        cudaEventElapsedTime(&ms, start, stop);
        kernel_ms_total += ms;

        cudaEventRecord(start);
        check_cuda(cudaMemcpy(h_C, d_C, bytes, cudaMemcpyDeviceToHost), "copy C D2H");
        cudaEventRecord(stop);
        cudaEventSynchronize(stop);
        cudaEventElapsedTime(&ms, start, stop);
        d2h_ms_total += ms;
    }

    float h2d_ms = h2d_ms_total / repeats;
    float kernel_ms = kernel_ms_total / repeats;
    float d2h_ms = d2h_ms_total / repeats;
    float total_ms = h2d_ms + kernel_ms + d2h_ms;

    double kernel_gflops = (2.0 * N * N * N) / ((kernel_ms / 1000.0) * 1e9);
    double total_gflops = (2.0 * N * N * N) / ((total_ms / 1000.0) * 1e9);
    double err = max_abs_error(N, h_C, h_ref);

    printf("version,N,repeats,h2d_ms,kernel_ms,d2h_ms,total_ms,kernel_GFLOPS,total_GFLOPS,max_abs_error\n");
    printf("cuda_%s,%d,%d,%.6f,%.6f,%.6f,%.6f,%.6f,%.6f,%.8f\n",
           version, N, repeats, h2d_ms, kernel_ms, d2h_ms, total_ms,
           kernel_gflops, total_gflops, err);

    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C); free(h_ref);
    return 0;
}

# Parte 6 — Compilação e execução CUDA

Caso esta célula falhe, verifique se o ambiente do Colab está com GPU ativada.

In [ ]:
!nvcc gemm_cuda.cu -O3 -o gemm_cuda

# Parte 7 — Experimentos automatizados em GPU

A próxima célula executa as versões CUDA para vários tamanhos de matriz e salva os resultados em `gpu_results.csv`.

In [ ]:
import subprocess
import pandas as pd
from io import StringIO

Ns = [128, 256, 512, 1024]
repeats = 5
rows = []

for N in Ns:
    for version in ["naive", "tiled"]:
        cmd = ["./gemm_cuda", version, str(N), str(repeats)]
        result = subprocess.run(cmd, capture_output=True, text=True)
        print(result.stdout)
        if result.returncode != 0:
            print(result.stderr)
            continue
        lines = result.stdout.strip().splitlines()
        if len(lines) >= 2:
            df = pd.read_csv(StringIO("\n".join(lines[-2:])))
            rows.append(df)

gpu_results = pd.concat(rows, ignore_index=True)
gpu_results.to_csv("gpu_results.csv", index=False)
gpu_results

# Parte 8 — Gráficos GPU

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

gpu_results = pd.read_csv("gpu_results.csv")
gpu_results

In [ ]:
for N in sorted(gpu_results["N"].unique()):
    df = gpu_results[gpu_results["N"] == N]
    plt.figure(figsize=(8, 4))
    plt.bar(df["version"], df["kernel_ms"])
    plt.ylabel("Tempo do kernel (ms)")
    plt.title(f"Tempo do kernel CUDA — N={N}")
    plt.tight_layout()
    plt.show()

In [ ]:
for N in sorted(gpu_results["N"].unique()):
    df = gpu_results[gpu_results["N"] == N]
    plt.figure(figsize=(8, 4))
    plt.bar(df["version"], df["kernel_GFLOPS"])
    plt.ylabel("Kernel GFLOP/s")
    plt.title(f"Desempenho do kernel CUDA — N={N}")
    plt.tight_layout()
    plt.show()

# Parte 9 — Tabela final de comparação

Preencha a tabela final com os principais resultados obtidos.

| Versão | N | Tempo médio | GFLOP/s | Speedup | Eficiência | Observações |
|---|---:|---:|---:|---:|---:|---|
| CPU naive |  |  |  | 1,0 | 1,0 | Referência |
| CPU transposed |  |  |  |  |  | Localidade |
| CPU blocked |  |  |  |  |  | Cache |
| CPU OpenMP 2 threads |  |  |  |  |  | Threads |
| CPU OpenMP 4 threads |  |  |  |  |  | Threads |
| CUDA naive |  |  |  |  |  | Kernel simples |
| CUDA tiled |  |  |  |  |  | Memória compartilhada |

## Questões para discussão

Responda no relatório:

1. Por que a versão ingênua apresenta baixo desempenho?
2. O impacto da transposição de `B` foi significativo? Por quê?
3. Qual tamanho de bloco apresentou melhor desempenho? Por quê?
4. O compilador conseguiu vetorizar alguma parte do código? Use o relatório de vetorização como evidência.
5. O speedup com OpenMP foi próximo do ideal? Explique.
6. A eficiência diminuiu com mais threads? Por quê?
7. A GPU foi mais rápida considerando apenas o kernel?
8. A GPU foi mais rápida considerando o tempo total com cópias?
9. Qual foi a principal limitação observada: computação, memória, comunicação ou overhead?
10. Quais cuidados são necessários ao usar o Google Colab para avaliar desempenho?

# Parte 10 — Critérios de avaliação

## Implementação — 5 pontos

| Critério | Pontos |
|---|---:|
| Código sequencial correto e validação dos resultados | 1,0 |
| Otimização de localidade / transposição / blocagem | 1,0 |
| Análise de vetorização SIMD | 1,0 |
| Paralelização com OpenMP | 1,0 |
| Implementação CUDA com medição separada de cópia e kernel | 1,0 |

## Relatório — 5 pontos

| Critério | Pontos |
|---|---:|
| Metodologia experimental clara | 1,0 |
| Tabelas e gráficos bem apresentados | 1,0 |
| Cálculo correto de GFLOP/s, speedup e eficiência | 1,0 |
| Discussão técnica dos resultados | 1,5 |
| Conclusão crítica sobre limitações e trade-offs | 0,5 |